## Introduction to Data Ingestion

In [5]:
import os
from typing import List, Dict, Any
import pandas as pd

In [6]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print("Set up!")

Set up!


### Understanding Document Structure in Langchain

In [8]:
## create a simple document
doc = Document(
    page_content="This the main text content thatwill be embedded and searched.",
    metadata={
        "source": "example.txt",
        "page" : 1,
        "author": "Abhay Mishra",
        "date_created": "2024-06-01",
        "custom_field": "Custom value"
        }
    )
print("Document structure")
print(f"Content : {doc.page_content}")
print(f"Metadata : {doc.metadata}")

# why metadata matters

print("\n Metadata is crucial for :")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")

Document structure
Content : This the main text content thatwill be embedded and searched.
Metadata : {'source': 'example.txt', 'page': 1, 'author': 'Abhay Mishra', 'date_created': '2024-06-01', 'custom_field': 'Custom value'}

 Metadata is crucial for :
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


In [9]:
type(doc)

langchain_core.documents.base.Document

### Text Files (.txt) - The Simplest Case {#2-text-files}

In [10]:
## create a simple txt file
import os
os.makedirs("data/text_files", exist_ok=True)

In [11]:
sample_texts = {
    "data/text_files/python_intro.txt": """Python is a high-level, easy-to-learn programming language known for its clean and readable syntax. It was created by Guido van Rossum and first released in 1991. Python emphasizes simplicity, which makes it a great choice for beginners as well as experienced developers.

One of Python’s biggest strengths is its versatility. It is widely used in areas like web development, data analysis, artificial intelligence, automation, and scientific computing. Popular platforms and tools such as Django and TensorFlow are built using or support Python.

Python also has a large standard library and a huge community, which means there are many ready-made modules and resources available. This allows developers to build applications quickly without having to start from scratch.

Overall, Python’s simplicity, flexibility, and powerful capabilities make it one of the most popular programming languages in the world today.
    """,

    "data/text_files/machine_learning.txt": """Machine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from data and make decisions or predictions without being explicitly programmed for every task.

Instead of writing fixed rules, developers train ML models using large amounts of data. The system then identifies patterns and improves its performance over time. For example, a machine learning model can learn to recognize spam emails, recommend movies, or even detect diseases from medical images.

There are three main types of machine learning:

* **Supervised Learning**: The model is trained on labeled data (e.g., predicting house prices based on past data).
* **Unsupervised Learning**: The model finds patterns in unlabeled data (e.g., customer segmentation).
* **Reinforcement Learning**: The model learns by interacting with an environment and receiving rewards or penalties (used in robotics and gaming).

Popular tools and libraries used in ML include TensorFlow, Scikit-learn, and PyTorch.

Machine learning is widely used in real-world applications such as voice assistants, recommendation systems, fraud detection, self-driving cars, and more. It is one of the fastest-growing and most impactful areas in technology today.
    """
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created in 'data/text_files' directory.")

Sample text files created in 'data/text_files' directory.


### TextLoader - Read Single File

In [12]:
from langchain_community.document_loaders import TextLoader

## Loading a single text file
loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")
documents = loader.load()
print(type(documents))
print(documents)

<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python is a high-level, easy-to-learn programming language known for its clean and readable syntax. It was created by Guido van Rossum and first released in 1991. Python emphasizes simplicity, which makes it a great choice for beginners as well as experienced developers.\n\nOne of Python’s biggest strengths is its versatility. It is widely used in areas like web development, data analysis, artificial intelligence, automation, and scientific computing. Popular platforms and tools such as Django and TensorFlow are built using or support Python.\n\nPython also has a large standard library and a huge community, which means there are many ready-made modules and resources available. This allows developers to build applications quickly without having to start from scratch.\n\nOverall, Python’s simplicity, flexibility, and powerful capabilities make it one of the most popular programming languages i

### DirectoryLoader - Multiple Text Files

In [13]:
from langchain_community.document_loaders import DirectoryLoader

## load all text files in a directory

dir_loader = DirectoryLoader(
    "data/text_files", 
    glob="**/*.txt", ##Pattern to match all text files in the directory and subdirectories
    loader_cls=TextLoader, ## Specify the loader class to use for loading each file
    loader_kwargs={"encoding": "utf-8"}, ## Additional arguments to pass to the loader class
    show_progress=True
    )

documents = dir_loader.load()
print(f"\nLoaded {len(documents)} documents from directory.")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1} :")
    print(f"Source : {doc.metadata.get('source', 'N/A')}")
    print(f"Content : {doc.page_content[:100]}...") ## Print first 100 characters
    print(f"Metadata : {doc.metadata}")
    print(f"Length of content : {len(doc.page_content)} characters")

100%|██████████| 2/2 [00:00<00:00, 156.56it/s]


Loaded 2 documents from directory.

Document 1 :
Source : data\text_files\machine_learning.txt
Content : Machine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from da...
Metadata : {'source': 'data\\text_files\\machine_learning.txt'}
Length of content : 1234 characters

Document 2 :
Source : data\text_files\python_intro.txt
Content : Python is a high-level, easy-to-learn programming language known for its clean and readable syntax. ...
Metadata : {'source': 'data\\text_files\\python_intro.txt'}
Length of content : 921 characters


In [14]:
# Analysis
print("\nDirectoryLoader Characteristics :")
print("\nAdvantages :")
print(" - Loads multiple files from a directory and subdirectories at once.")
print(" - Supports glob patterns to filter specific file types.")
print(" - Progress tracking")
print(" - Recursive directory scanning")

print("\nDisadvantages :")
print(" - All Files must be of the same type or require the same loader.")
print(" - Limited error handling for individual file issues (e.g., encoding errors) which may halt the entire loading process.")
print(" - Can be memory intensive if loading a large number of files or very large files at once.")


DirectoryLoader Characteristics :

Advantages :
 - Loads multiple files from a directory and subdirectories at once.
 - Supports glob patterns to filter specific file types.
 - Progress tracking
 - Recursive directory scanning

Disadvantages :
 - All Files must be of the same type or require the same loader.
 - Limited error handling for individual file issues (e.g., encoding errors) which may halt the entire loading process.
 - Can be memory intensive if loading a large number of files or very large files at once.


## Text Spliting Stratergies

In [16]:
## Different text splitting strategies
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

print(documents)

[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from data and make decisions or predictions without being explicitly programmed for every task.\n\nInstead of writing fixed rules, developers train ML models using large amounts of data. The system then identifies patterns and improves its performance over time. For example, a machine learning model can learn to recognize spam emails, recommend movies, or even detect diseases from medical images.\n\nThere are three main types of machine learning:\n\n* **Supervised Learning**: The model is trained on labeled data (e.g., predicting house prices based on past data).\n* **Unsupervised Learning**: The model finds patterns in unlabeled data (e.g., customer segmentation).\n* **Reinforcement Learning**: The model learns by interacting with an environment and receiving rewards or penalties (used in robotics and gaming

In [29]:
### Method 1 - Character Text Splitter
text = documents[0].page_content
char_splitter = CharacterTextSplitter(
    separator="\n",     # split on newlines
    chunk_size=200,     # Max chunk size in characters
    chunk_overlap=20,   # Overlap between chunks in characters
    length_function=len # How to measure chunk size (in this case, number of characters)
)

char_chunks = char_splitter.split_text(text)
print(f"Number of chunks created: {len(char_chunks)}")
for i, chunk in enumerate(char_chunks):
    print(f"\nChunk {i+1} :")
    print(f"Content : {chunk}") # Print first 100 characters of the chunk
    print(f"Chunk length : {len(chunk)} characters")

Created a chunk of size 300, which is longer than the specified 200
Created a chunk of size 233, which is longer than the specified 200


Number of chunks created: 7

Chunk 1 :
Content : Machine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from data and make decisions or predictions without being explicitly programmed for every task.
Chunk length : 188 characters

Chunk 2 :
Content : Instead of writing fixed rules, developers train ML models using large amounts of data. The system then identifies patterns and improves its performance over time. For example, a machine learning model can learn to recognize spam emails, recommend movies, or even detect diseases from medical images.
Chunk length : 300 characters

Chunk 3 :
Content : There are three main types of machine learning:
* **Supervised Learning**: The model is trained on labeled data (e.g., predicting house prices based on past data).
Chunk length : 163 characters

Chunk 4 :
Content : * **Unsupervised Learning**: The model finds patterns in unlabeled data (e.g., customer segmentation).
Chunk length : 102 characters

Chunk 5 :


In [ ]:
## Method 2 - Recursive Character Text Splitter (RECOMMENDED)
print("\nUsing Recursive Character Text Splitter")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], # Try splitting on double newlines, then single newlines, then spaces, and finally characters
    chunk_size=200,     # Max chunk size in characters
    chunk_overlap=20,   # Overlap between chunks in characters
    length_function=len # How to measure chunk size (in this case, number of characters)
)

recursive_chunks = recursive_splitter.split_text(text)

print(f"Number of chunks created: {len(recursive_chunks)}")
for i, chunk in enumerate(recursive_chunks):
    print(f"\nChunk {i+1} :")
    print(f"Content : {chunk}") # Print first 100 characters of the chunk
    print(f"Chunk length : {len(chunk)} characters")

In [34]:
# Method 3 - Token Text Splitter
print("\nUsing Token Text Splitter")

token_splitter = TokenTextSplitter(
    chunk_size=50,     # Max chunk size in tokens
    chunk_overlap=5    # Overlap between chunks in tokens
)

token_chunks = token_splitter.split_text(text)
print(f"Number of chunks created: {len(token_chunks)}")
for i, chunk in enumerate(token_chunks):
    print(f"\nChunk {i+1} :")
    print(f"Content : {chunk}") # Print first 100 characters of the chunk
    print(f"Chunk length : {len(chunk)} tokens")


Using Token Text Splitter
Number of chunks created: 6

Chunk 1 :
Content : Machine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from data and make decisions or predictions without being explicitly programmed for every task.

Instead of writing fixed rules, developers train ML models using large amounts of data. The
Chunk length : 281 tokens

Chunk 2 :
Content :  amounts of data. The system then identifies patterns and improves its performance over time. For example, a machine learning model can learn to recognize spam emails, recommend movies, or even detect diseases from medical images.

There are three main types of machine
Chunk length : 269 tokens

Chunk 3 :
Content :  three main types of machine learning:

* **Supervised Learning**: The model is trained on labeled data (e.g., predicting house prices based on past data).
* **Unsupervised Learning**: The model finds patterns in
Chunk length : 212 tokens

Chunk 4 :
Content :  The model finds p

In [35]:
# Comparison of splitting strategies
print("\nComparison of Text Splitting Strategies :")
print("\nCharacter Text Splitter :")
print("Simple and predictable splitting based on characters.")
print("Good for structured text with clear delimiters (e.g., newlines).")
print("May split in the middle of sentences or words if chunk size is not carefully chosen.")
print("Used when: Text has clear delimiters and you want control over character-based chunking.")

print("\nRecursive Character Text Splitter :")
print("More intelligent splitting that tries multiple separators.")
print("Respects natural text boundaries (e.g., paragraphs, sentences) when possible.")
print("Best general-purpose splitter for most text types.")
print("Slightly more complex but produces more coherent chunks.")
print("Used when: You want better chunk coherence and your text has natural delimiters.")

print("\nToken Text Splitter :")
print("Splits based on tokens rather than characters.")
print("Respects token boundaries, which can be important for language models.")
print("More accurate for embedding and LLM applications where token limits matter.")
print("Slower than character-based splitters due to tokenization overhead.")
print("Used when: You need precise control over token counts for embedding or LLM input, and are okay with the additional complexity.")


Comparison of Text Splitting Strategies :

Character Text Splitter :
Simple and predictable splitting based on characters.
Good for structured text with clear delimiters (e.g., newlines).
May split in the middle of sentences or words if chunk size is not carefully chosen.
Used when: Text has clear delimiters and you want control over character-based chunking.

Recursive Character Text Splitter :
More intelligent splitting that tries multiple separators.
Respects natural text boundaries (e.g., paragraphs, sentences) when possible.
Best general-purpose splitter for most text types.
Slightly more complex but produces more coherent chunks.
Used when: You want better chunk coherence and your text has natural delimiters.

Token Text Splitter :
Splits based on tokens rather than characters.
Respects token boundaries, which can be important for language models.
More accurate for embedding and LLM applications where token limits matter.
Slower than character-based splitters due to tokenization